# Looker Folder Permissions & Content Isolation Lifecycle (Single Shared Folder)

Moves a set of dashboards/looks into **one shared isolation folder**, locks every access entry on that folder down to view-only, and later moves everything back to exactly where it came from.

Each phase (Snapshot / Isolate / Restore) is its own set of cells so you can run and inspect one phase at a time. State is passed between phases through two JSON files on disk — `content_snapshot.json` (Phase 1) and `isolation_state.json` (Phase 2) — rather than in-memory variables, so Phase 3 can be run in a completely separate session (a different day, a different machine, even a different person) as long as those two files are available.

Requires: `pip install looker_sdk`, and a `looker.ini` credentials file in this notebook's working directory (see `setup_credentials.py`).

In [ ]:
from __future__ import annotations

import json
import logging
import os
from typing import Any, Dict, List, Optional
import looker_sdk
from looker_sdk import error, models40

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("looker_lifecycle")


## Connect to Looker

Reads credentials from `looker.ini` in the current working directory.

In [ ]:
sdk = looker_sdk.init40()


## Phase 1: Snapshot

Records current locations (`folder_id`) of target dashboards and looks.

`DASHBOARD_IDS` accepts either real numeric dashboard IDs or dashboard **slugs** (e.g. from a dashboard URL) — anything that isn't a plain number is resolved to its real ID via `search_dashboards(slug=...)` first. `LOOK_IDS` must always be real look IDs: Looker has no slug field or slug-based lookup for looks.

Function definitions — run these cells once, then use the cells below to actually capture a snapshot.

In [ ]:
def _resolve_dashboard_id(sdk: looker_sdk.methods40.Looker40SDK, dashboard_id_or_slug: str) -> str:
    """Resolves a dashboard identifier that may be a numeric ID or a slug into a real dashboard ID.

    Plain digit strings (e.g. "1342") are assumed to already be real IDs and are returned as-is.
    Anything else is treated as a slug and resolved via search_dashboards(slug=...) -- Looker
    does not offer a direct "get by slug" lookup, only a search filter. Looks have no equivalent
    slug field/search filter in the Looker API, so this resolution only applies to dashboards.
    """
    value = dashboard_id_or_slug.strip()
    if value.isdigit():
        return value

    matches = sdk.search_dashboards(slug=value, fields="id,title")
    if not matches:
        raise ValueError(f"No dashboard found for slug '{value}'")
    if len(matches) > 1:
        logger.warning(
            "Slug '%s' matched %d dashboards; using the first match (ID: %s, title: '%s').",
            value, len(matches), matches[0].id, matches[0].title,
        )
    resolved_id = str(matches[0].id)
    logger.info("Resolved dashboard slug '%s' -> ID %s", value, resolved_id)
    return resolved_id


def snapshot_content_locations(
    sdk: looker_sdk.methods40.Looker40SDK,
    dashboard_ids: Optional[List[str]] = None,
    look_ids: Optional[List[str]] = None,
) -> List[Dict[str, Any]]:
    """Captures the current folder locations of specified dashboards and looks.

    Args:
        sdk: Authenticated Looker 4.0 SDK instance.
        dashboard_ids: List of dashboard IDs OR slugs to snapshot (both are accepted -- see
            _resolve_dashboard_id).
        look_ids: List of look IDs to snapshot. Looks must use their real ID; Looker has no
            slug field/lookup for looks.

    Returns:
        A list of dictionaries containing asset metadata and original_folder_id.
    """
    dashboard_ids = dashboard_ids or []
    look_ids = look_ids or []
    snapshot: List[Dict[str, Any]] = []

    logger.info("Starting snapshot for %d dashboard(s) and %d look(s)...", len(dashboard_ids), len(look_ids))

    for d_id in dashboard_ids:
        d_id_clean = str(d_id).strip()
        if not d_id_clean:
            continue
        try:
            resolved_id = _resolve_dashboard_id(sdk, d_id_clean)
            d = sdk.dashboard(dashboard_id=resolved_id, fields="id,title,folder_id")
            snapshot.append({
                "type": "dashboard",
                "id": str(d.id),
                "title": d.title,
                "original_folder_id": str(d.folder_id),
            })
            logger.info("Recorded Dashboard '%s' (ID: %s) -> Folder ID: %s", d.title, d.id, d.folder_id)
        except Exception as e:
            logger.error("Failed to query dashboard '%s': %s", d_id_clean, e)
            raise

    for l_id in look_ids:
        l_id_clean = str(l_id).strip()
        if not l_id_clean:
            continue
        try:
            l = sdk.look(look_id=l_id_clean, fields="id,title,folder_id")
            snapshot.append({
                "type": "look",
                "id": str(l.id),
                "title": l.title,
                "original_folder_id": str(l.folder_id),
            })
            logger.info("Recorded Look '%s' (ID: %s) -> Folder ID: %s", l.title, l.id, l.folder_id)
        except Exception as e:
            logger.error("Failed to query look ID '%s': %s", l_id_clean, e)
            raise

    return snapshot


In [ ]:
def snapshot_folder_permissions(
    sdk: looker_sdk.methods40.Looker40SDK,
    folder_id: str,
) -> Dict[str, Any]:
    """Captures folder permission metadata and access rules (for audit/verification)."""
    folder = sdk.folder(folder_id=str(folder_id), fields="id,name,content_metadata_id,parent_id")
    cm_id = str(folder.content_metadata_id)
    cm = sdk.content_metadata(content_metadata_id=cm_id)
    accesses = sdk.all_content_metadata_accesses(content_metadata_id=cm_id)

    return {
        "folder_id": str(folder.id),
        "name": folder.name,
        "parent_id": str(folder.parent_id),
        "content_metadata_id": cm_id,
        "inherits": cm.inherits,
        "access_rules": [
            {
                "id": str(a.id),
                "group_id": str(a.group_id) if a.group_id else None,
                "user_id": str(a.user_id) if a.user_id else None,
                "permission_type": a.permission_type,  # 'view' or 'edit'
            }
            for a in accesses
        ],
    }


### Run Phase 1

Edit `DASHBOARD_IDS` / `LOOK_IDS` below, then run this cell to capture the snapshot and see it printed inline.

In [ ]:
DASHBOARD_IDS = ["1342", "8hJ5ucbqnyNyMxjEpSfsiP", "1344"]  # real dashboard IDs, or slugs (e.g. "abcXYZ123") -- both work
LOOK_IDS = ["378", "379", "380"]                # must be real look IDs -- no slug support for looks

content_snapshot = snapshot_content_locations(sdk, dashboard_ids=DASHBOARD_IDS, look_ids=LOOK_IDS)
content_snapshot


Save the snapshot to disk. Phase 2 and Phase 3 read `content_snapshot` from this file rather than from the in-memory variable, so nothing here depends on keeping this notebook session alive — keep `content_snapshot.json` somewhere handy until you've completed Phase 3.

In [ ]:
SNAPSHOT_FILE = "content_snapshot.json"

with open(SNAPSHOT_FILE, "w", encoding="utf-8") as f:
    json.dump(content_snapshot, f, indent=2)

logger.info("Snapshot saved successfully to %s", SNAPSHOT_FILE)


## Phase 2: Isolate

Creates a **single** subfolder under one shared `PARENT_FOLDER_ID`, breaks its permission inheritance, and locks **every** access entry that ends up on the new folder down to view-only (not just one target group). The original folder's own permissions are never permanently changed.

**Ancestor conflict handling:** Looker refuses to downgrade a group/user's access below what they already have on the folder's DIRECT parent. If that happens here, the code walks up the folder tree starting from `PARENT_FOLDER_ID` to find whichever ancestor(s) explicitly grant that higher access, snapshots and persists each one's full permission state to disk (`permission_snapshots/`) before touching it, temporarily downgrades them (root-most first), performs the real downgrade on the new isolation folder, then restores every touched ancestor back to its original state (nearest-first) — guaranteed via try/finally even if something fails partway. A failed restore is logged loudly with the snapshot file path rather than failing silently.

This ancestor downgrade-and-restore happens entirely within this phase — by the time this cell finishes, any ancestor folder touched has already been put back exactly as it was. The isolation folder itself keeps its explicit `view` setting regardless, since it's no longer inheriting from the parent. That means Phase 3 doesn't need any permission-revert step of its own — it only needs to move content back and delete the isolation folder.

Function definitions:

In [ ]:
PERMISSION_SNAPSHOT_DIR = "permission_snapshots"


def _get_folder_state(sdk: looker_sdk.methods40.Looker40SDK, folder_id: str) -> Dict[str, Any]:
    """Fetches a folder's parent_id, content_metadata_id, and current access records."""
    folder = sdk.folder(folder_id=str(folder_id), fields="id,name,parent_id,content_metadata_id")
    cm_id = str(folder.content_metadata_id)
    accesses = sdk.all_content_metadata_accesses(content_metadata_id=cm_id)
    return {
        "folder_id": str(folder.id),
        "folder_name": folder.name,
        "parent_id": str(folder.parent_id) if folder.parent_id else None,
        "content_metadata_id": cm_id,
        "accesses": accesses,
    }


def _find_own_access_record(accesses, group_id: Optional[str], user_id: Optional[str]):
    """Finds the access record on this folder belonging to the same principal (group or user)."""
    for a in accesses:
        if group_id is not None and a.group_id is not None and str(a.group_id) == str(group_id):
            return a
        if user_id is not None and a.user_id is not None and str(a.user_id) == str(user_id):
            return a
    return None


def find_blocking_ancestor_chain(
    sdk: looker_sdk.methods40.Looker40SDK,
    start_folder_id: str,
    group_id: Optional[str],
    user_id: Optional[str],
) -> List[Dict[str, Any]]:
    """Walks up from start_folder_id to the root folder, collecting every ancestor that has
    its own explicit (non-'view') access record for this same principal (group or user).

    Only ancestors with an explicit record are actionable (there's nothing to downgrade on
    a folder that's purely inheriting). Returns the chain ordered nearest-to-root.
    """
    chain: List[Dict[str, Any]] = []
    current_id: Optional[str] = str(start_folder_id)

    while current_id:
        state = _get_folder_state(sdk, current_id)
        own_record = _find_own_access_record(state["accesses"], group_id, user_id)
        if own_record is not None and own_record.permission_type.value != "view":
            chain.append({
                "folder_id": state["folder_id"],
                "folder_name": state["folder_name"],
                "content_metadata_id": state["content_metadata_id"],
                "access_id": str(own_record.id),
                "original_permission": own_record.permission_type.value,
            })
        current_id = state["parent_id"]

    return chain


def snapshot_and_persist_folder_permissions(sdk: looker_sdk.methods40.Looker40SDK, folder_id: str) -> str:
    """Captures a folder's full permission state (every access record, not just one
    principal) and writes it to disk BEFORE any modification, so there's a durable
    record to restore from even if the process crashes mid-operation."""
    snap = snapshot_folder_permissions(sdk, folder_id)
    snap_serializable = dict(snap)
    snap_serializable["access_rules"] = [
        {**rule, "permission_type": rule["permission_type"].value if rule["permission_type"] else None}
        for rule in snap["access_rules"]
    ]
    os.makedirs(PERMISSION_SNAPSHOT_DIR, exist_ok=True)
    path = os.path.join(PERMISSION_SNAPSHOT_DIR, f"folder_{folder_id}_permissions.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(snap_serializable, f, indent=2)
    logger.info("Persisted permission snapshot for folder %s to %s", folder_id, path)
    return path


def downgrade_with_parent_conflict_resolution(
    sdk: looker_sdk.methods40.Looker40SDK,
    origin_folder_id: str,
    group_id: Optional[str],
    user_id: Optional[str],
    target_access_id: str,
) -> None:
    """Downgrades target_access_id to 'view'. Looker refuses this if the same principal
    (group/user) has a higher effective permission on the DIRECT parent folder.

    If blocked, this walks up the ancestor chain from origin_folder_id, snapshots and
    persists every ancestor it's about to touch, temporarily downgrades each blocking
    ancestor (farthest/root first, since downgrading one requires its own parent to
    already be unblocked), retries the original downgrade, then restores every touched
    ancestor back to its original permission (nearest first) -- guaranteed via
    try/finally, even if something fails partway. A failed restore is logged loudly
    with the snapshot file path, rather than failing silently.
    """
    try:
        sdk.update_content_metadata_access(
            content_metadata_access_id=target_access_id,
            body=models40.ContentMetaGroupUser(permission_type="view"),
        )
        return
    except error.SDKError as e:
        if "Can't downgrade to view when have edit on parent" not in str(e):
            raise
        logger.info(
            "Direct downgrade blocked by parent access -- walking up ancestor chain from folder %s...",
            origin_folder_id,
        )

    chain = find_blocking_ancestor_chain(sdk, origin_folder_id, group_id, user_id)

    if not chain:
        logger.error(
            "Downgrade blocked but no explicit blocking ancestor found above folder %s for "
            "group_id=%s/user_id=%s -- this principal's access may not be folder-based "
            "(e.g. a broader role/permission). Cannot auto-resolve.",
            origin_folder_id, group_id, user_id,
        )
        raise

    logger.info(
        "Ancestor chain to temporarily downgrade (nearest to farthest): %s",
        [f"{c['folder_name']} ({c['folder_id']}): {c['original_permission']}" for c in chain],
    )

    for c in chain:
        snapshot_and_persist_folder_permissions(sdk, c["folder_id"])

    touched: List[Dict[str, Any]] = []
    try:
        # Downgrade farthest (root) first, down to nearest -- required order, since
        # downgrading a folder is blocked while ITS OWN parent still grants more.
        for c in reversed(chain):
            logger.info(
                "Temporarily downgrading '%s' (ID: %s) from '%s' to 'view'...",
                c["folder_name"], c["folder_id"], c["original_permission"],
            )
            sdk.update_content_metadata_access(
                content_metadata_access_id=c["access_id"],
                body=models40.ContentMetaGroupUser(permission_type="view"),
            )
            touched.append(c)

        # Retry the original target downgrade now that ancestors are unblocked.
        sdk.update_content_metadata_access(
            content_metadata_access_id=target_access_id,
            body=models40.ContentMetaGroupUser(permission_type="view"),
        )
        logger.info("Target access %s successfully downgraded to view after ancestor resolution.", target_access_id)

    finally:
        # Restore nearest-to-root order (upgrades are never blocked by this rule).
        for c in reversed(touched):
            try:
                logger.info(
                    "Restoring '%s' (ID: %s) back to '%s'...",
                    c["folder_name"], c["folder_id"], c["original_permission"],
                )
                sdk.update_content_metadata_access(
                    content_metadata_access_id=c["access_id"],
                    body=models40.ContentMetaGroupUser(permission_type=c["original_permission"]),
                )
            except Exception as restore_error:
                logger.error(
                    "FAILED TO RESTORE folder '%s' (ID: %s) to '%s' -- MANUAL FIX REQUIRED IN LOOKER. "
                    "Original state is saved in %s/folder_%s_permissions.json. Error: %s",
                    c["folder_name"], c["folder_id"], c["original_permission"],
                    PERMISSION_SNAPSHOT_DIR, c["folder_id"], restore_error,
                )


def create_isolated_view_folder(
    sdk: looker_sdk.methods40.Looker40SDK,
    folder_name: str,
    parent_folder_id: str,
    content_snapshot: List[Dict[str, Any]],
) -> str:
    """Creates a single subfolder under parent_folder_id, breaks inheritance, locks every
    cloned access entry to 'view' (resolving any ancestor conflicts along the way), and
    relocates all snapshotted content into it.

    Args:
        sdk: Authenticated Looker 4.0 SDK instance.
        folder_name: Name for the temporary staging folder.
        parent_folder_id: Folder ID under which to create the new isolation folder.
        content_snapshot: List of items to relocate into this folder.

    Returns:
        The ID of the newly created and populated folder.
    """
    logger.info("Creating new folder '%s' under parent ID %s...", folder_name, parent_folder_id)

    # 1. Create the new folder
    new_folder = sdk.create_folder(
        body=models40.CreateFolder(name=folder_name, parent_id=str(parent_folder_id))
    )
    new_folder_id = str(new_folder.id)
    cm_id = str(new_folder.content_metadata_id)
    logger.info("Created folder ID: %s (content_metadata_id: %s)", new_folder_id, cm_id)

    # 2. Sever permission inheritance
    # Note: this clones the parent folder's access entries (groups/users) down onto this folder.
    logger.info("Decoupling permission inheritance (inherits=False) on content metadata %s...", cm_id)
    sdk.update_content_metadata(
        content_metadata_id=cm_id,
        body=models40.WriteContentMeta(inherits=False),
    )

    # 3. Lock every cloned access entry to view-only
    existing_accesses = sdk.all_content_metadata_accesses(content_metadata_id=cm_id)
    for access in existing_accesses:
        # access.permission_type is a PermissionType enum member, not a plain string --
        # compare against .value so already-'view' entries are correctly skipped.
        current_permission = access.permission_type.value if access.permission_type is not None else None
        if current_permission != "view":
            logger.info(
                "Downgrading access record %s (group_id=%s, user_id=%s) from '%s' to 'view'...",
                access.id, access.group_id, access.user_id, current_permission,
            )
            downgrade_with_parent_conflict_resolution(
                sdk=sdk,
                origin_folder_id=parent_folder_id,
                group_id=str(access.group_id) if access.group_id else None,
                user_id=str(access.user_id) if access.user_id else None,
                target_access_id=str(access.id),
            )
        else:
            logger.info(
                "Access record %s (group_id=%s, user_id=%s) is already 'view'.",
                access.id, access.group_id, access.user_id,
            )

    # 4. Move content into the isolated folder
    logger.info("Relocating %d asset(s) to folder ID %s...", len(content_snapshot), new_folder_id)
    for item in content_snapshot:
        item_id = str(item["id"])
        item_type = item["type"]
        item_title = item.get("title", "Untitled")

        if item_type == "dashboard":
            sdk.update_dashboard(
                dashboard_id=item_id,
                body=models40.WriteDashboard(folder_id=new_folder_id),
            )
            logger.info("Moved Dashboard '%s' (ID: %s) -> New Folder: %s", item_title, item_id, new_folder_id)
        elif item_type == "look":
            sdk.update_look(
                look_id=item_id,
                body=models40.WriteLookWithQuery(folder_id=new_folder_id),
            )
            logger.info("Moved Look '%s' (ID: %s) -> New Folder: %s", item_title, item_id, new_folder_id)
        else:
            logger.warning("Unrecognized asset type '%s' (ID: %s). Skipped.", item_type, item_id)

    logger.info("Content isolation complete. Folder ID: %s", new_folder_id)
    return new_folder_id


### Run Phase 2

Edit `PARENT_FOLDER_ID` and `FOLDER_NAME` below. This reads `content_snapshot` from `content_snapshot.json` on disk (not from the in-memory Phase 1 variable), so it works the same whether Phase 1 just ran in this session or ran a week ago in a different one. Every access entry that ends up on the new isolation folder is locked to view-only — there's no separate target group to configure.

In [ ]:
PARENT_FOLDER_ID = "1"  # edit: folder ID under which the isolation folder is created (default CLI value: "1" / Shared)
FOLDER_NAME = "Restricted Review Folder"

with open("content_snapshot.json", "r", encoding="utf-8") as f:
    content_snapshot = json.load(f)

isolation_folder_id = create_isolated_view_folder(
    sdk=sdk,
    folder_name=FOLDER_NAME,
    parent_folder_id=PARENT_FOLDER_ID,
    content_snapshot=content_snapshot,
)
print(f"ISOLATION_FOLDER_ID={isolation_folder_id}")


Save the isolation folder ID to disk. Phase 3 reads this from `isolation_state.json` rather than from the in-memory variable — keep this file handy alongside `content_snapshot.json` until you've run Phase 3.

In [ ]:
ISOLATION_STATE_FILE = "isolation_state.json"

with open(ISOLATION_STATE_FILE, "w", encoding="utf-8") as f:
    json.dump({"isolation_folder_id": isolation_folder_id}, f, indent=2)

logger.info("Isolation folder ID saved successfully to %s", ISOLATION_STATE_FILE)


## Phase 3: Restore & Cleanup

Relocates content back to their original folders using the snapshot, verifies the single temporary folder is empty, and deletes it.

**Note:** this permanently deletes the isolation folder — only run this cell when you actually mean to restore and clean up.

Function definition:

In [ ]:
def restore_content_and_cleanup(
    sdk: looker_sdk.methods40.Looker40SDK,
    temp_folder_id: str,
    content_snapshot: List[Dict[str, Any]],
) -> None:
    """Restores content back to their original folder IDs and safely deletes the temporary folder.

    Args:
        sdk: Authenticated Looker 4.0 SDK instance.
        temp_folder_id: ID of the temporary isolation folder to clean up.
        content_snapshot: Snapshot containing the original_folder_id for each item.
    """
    temp_folder_id_clean = str(temp_folder_id).strip()
    logger.info("Starting restoration for %d asset(s)...", len(content_snapshot))

    # 1. Restore all items to their original folders
    for item in content_snapshot:
        item_id = str(item["id"])
        orig_folder_id = str(item["original_folder_id"])
        item_type = item["type"]
        item_title = item.get("title", "Untitled")

        if item_type == "dashboard":
            sdk.update_dashboard(
                dashboard_id=item_id,
                body=models40.WriteDashboard(folder_id=orig_folder_id),
            )
            logger.info("Restored Dashboard '%s' (ID: %s) -> Original Folder: %s", item_title, item_id, orig_folder_id)
        elif item_type == "look":
            sdk.update_look(
                look_id=item_id,
                body=models40.WriteLookWithQuery(folder_id=orig_folder_id),
            )
            logger.info("Restored Look '%s' (ID: %s) -> Original Folder: %s", item_title, item_id, orig_folder_id)

    # 2. Pre-flight verification before deleting folder
    logger.info("Verifying temporary folder %s is empty...", temp_folder_id_clean)
    remaining_dashboards = sdk.folder_dashboards(folder_id=temp_folder_id_clean)
    remaining_looks = sdk.folder_looks(folder_id=temp_folder_id_clean)

    if remaining_dashboards or remaining_looks:
        msg = (
            f"ABORTING DELETION: Folder {temp_folder_id_clean} still contains "
            f"{len(remaining_dashboards)} dashboard(s) and {len(remaining_looks)} look(s)!"
        )
        logger.error(msg)
        raise RuntimeError(msg)

    # 3. Permanently delete the temporary folder
    logger.info("Deleting empty temporary folder %s...", temp_folder_id_clean)
    sdk.delete_folder(folder_id=temp_folder_id_clean)
    logger.info("Successfully deleted temporary folder %s. Restoration finished.", temp_folder_id_clean)


### Run Phase 3

Reads `content_snapshot` from `content_snapshot.json` and `isolation_folder_id` from `isolation_state.json` on disk — not from in-memory variables. This means Phase 3 can be run in a brand new session (a different day, a different machine, even a different person), as long as both files are available.

In [ ]:
with open("content_snapshot.json", "r", encoding="utf-8") as f:
    content_snapshot = json.load(f)
with open("isolation_state.json", "r", encoding="utf-8") as f:
    isolation_folder_id = json.load(f)["isolation_folder_id"]

restore_content_and_cleanup(
    sdk=sdk,
    temp_folder_id=isolation_folder_id,
    content_snapshot=content_snapshot,
)
print("RESTORE_COMPLETED=TRUE")
